# Q4: Transglial Mitochondrial Communication — LPC DBiT Spatial Data

This notebook asks whether Cx47 (GJC2) correlates with evidence of transglial mitochondrial communication across the LPC demyelination time course (Day 5 → Day 10 → Day 21).

**Dataset:** LPC DBiT spatial RNA (Day 5, 10, 21), single S1 section per timepoint  
**Strategy:** Each DBiT spot integrates signal from multiple cells. We score lineage programs per spot to identify oligodendrocyte-enriched vs astrocyte/microglia-enriched spots, then ask:
1. Do GJC2 levels correlate with mitochondrial programs in oligo-enriched spots?
2. Do mitochondrial programs in oligo spots co-vary with astrocyte/microglia programs in the same spatial region (proxy for transglial coupling)?
3. Do vesicle trafficking, EV markers, and TNT-related genes track with GJC2 and mito programs?
4. Does any coupling signal change across the three time points?

> **Note:** Single section per timepoint — all peak-stage calls are exploratory trends.

In [1]:
from pathlib import Path
import os
import sys

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from IPython.display import Markdown, display

CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(
    (p for p in CANDIDATES if (p / "src").exists() and (p / "config").exists()), None
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repository root.")

MPL_CACHE_DIR = REPO_ROOT / ".cache" / "matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from cx47_oligo.gene_panels import GENE_PANELS
from cx47_oligo.exports import ensure_results_dir, save_current_figure, save_json, save_table
from cx47_oligo.h5ad_tools import (
    adata_overview,
    expression_frame,
    panel_availability_table,
    resolve_gene_symbols,
    score_gene_panel,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#fbfcfd",
    "axes.edgecolor": "#d7dde2",
    "grid.color": "#dde4ea",
    "grid.alpha": 0.75,
})

In [2]:
DATASET_PATHS = {
    5:  Path("/Volumes/processing2/KaroSpaceDataWrangle/dbit-data/DBiT_Di/LPC5_S1_RNA.h5ad"),
    10: Path("/Volumes/processing2/KaroSpaceDataWrangle/dbit-data/DBiT_Di/LPC10_S1_RNA.h5ad"),
    21: Path("/Volumes/processing2/KaroSpaceDataWrangle/dbit-data/DBiT_Di/LPC21_S1_RNA.h5ad"),
}
for day, path in DATASET_PATHS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing LPC Day {day}: {path}")

RESULTS_DIR = ensure_results_dir(REPO_ROOT, "04_transglial_mito_communication", "lpc_dbit_multi")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_GENES = ["GJC2", "GJB1", "GJA1"]
TIME_ORDER = [f"Day {d}" for d in sorted(DATASET_PATHS)]

# Panels grouped by function
LINEAGE_PANELS = ["oligodendrocyte_identity", "astrocyte_identity", "microglial_activation"]
MITO_PANELS = [
    "mitochondrial_oxphos",
    "mitochondrial_biogenesis",
    "mitochondrial_dynamics",
    "mitochondrial_membrane_transport",
    "mitophagy_stress",
    "mitochondrial_trafficking",
    "metabolic_coupling",
]
TRANSFER_PANELS = [
    "vesicle_trafficking",
    "ev_markers",
    "tnt_core",
    "actin_cytoskeleton",
    "mdv_lysosomal",
    "autophagy_stress",
]
ALL_PANELS = list(dict.fromkeys(LINEAGE_PANELS + MITO_PANELS + TRANSFER_PANELS + ["panglial_connexins"]))

print("Results:", RESULTS_DIR)
for day, path in sorted(DATASET_PATHS.items()):
    print(f"  Day {day}: {path.name}")

Results: /Users/chrislangseth/work/karolinska_institutet/projects/cx47-oligo/results/04_transglial_mito_communication/lpc_dbit_multi
  Day 5: LPC5_S1_RNA.h5ad
  Day 10: LPC10_S1_RNA.h5ad
  Day 21: LPC21_S1_RNA.h5ad


In [3]:
import gc

# Collect every gene needed across all panels + target genes.
# We resolve var-names from the first file (all three share the same gene space)
# and subset *each* file before concat — so we never hold more than
# 3 × (small subset) in memory instead of 4 × (23k-gene full matrix).
_first = list(sorted(DATASET_PATHS.items()))[0][1]
_probe = __import__("anndata").read_h5ad(_first)
_probe.var_names_make_unique()

_query_genes = list(dict.fromkeys(
    [g for panel in ALL_PANELS for g in GENE_PANELS[panel]] + list(TARGET_GENES)
))
_, _, _gene_to_var = resolve_gene_symbols(_probe, _query_genes)
_keep_vars = list(dict.fromkeys(_gene_to_var.values()))
del _probe
gc.collect()
print(f"Gene subset: keeping {len(_keep_vars)} vars")

adatas = []
for day, path in sorted(DATASET_PATHS.items()):
    a = ad.read_h5ad(path)
    a.obs_names_make_unique()
    a.var_names_make_unique()
    # Subset immediately — before appending — so concat never sees full matrices
    a = a[:, _keep_vars].copy()
    a.obs["day_post_lpc"] = day
    a.obs["timepoint_label"] = f"Day {day}"
    a.obs["sample_id"] = path.stem.replace("_RNA", "")
    a.uns.setdefault("source_dataset", {})["value_type"] = "normalized"
    adatas.append(a)
    print(f"  Day {day}: {a.n_obs} spots × {a.n_vars} genes")

adata = ad.concat(adatas, join="inner", merge="same", index_unique="__")
adata.uns["source_dataset"] = {"value_type": "normalized"}
del adatas
gc.collect()

display(adata_overview(adata))

coverage = panel_availability_table(adata, ALL_PANELS)
display(coverage)
save_table(coverage, RESULTS_DIR / "panel_coverage.csv", index=False)

gene_status_rows = []
for gene in TARGET_GENES:
    present, _, gtv = resolve_gene_symbols(adata, [gene])
    gene_status_rows.append({"gene": gene, "resolved": gtv.get(gene, ""), "present": bool(present)})
display(pd.DataFrame(gene_status_rows))

save_json(
    {
        "dataset_paths": {str(d): str(p) for d, p in DATASET_PATHS.items()},
        "results_dir": str(RESULTS_DIR),
        "time_order": TIME_ORDER,
        "lineage_panels": LINEAGE_PANELS,
        "mito_panels": MITO_PANELS,
        "transfer_panels": TRANSFER_PANELS,
        "n_vars_kept": len(_keep_vars),
    },
    RESULTS_DIR / "run_metadata.json",
)


,metric,value
0,n_obs,25401
1,n_vars,23364
2,obs_columns,10
3,var_columns,0
4,layers,(none)
5,obsm,(none)
6,uns_keys,source_dataset
7,raw_present,True


,panel,panel_size,present_count,missing_count,coverage,present_genes,missing_genes
0,actin_cytoskeleton,8,8,0,1.000,"ACTB, ACTG1, ACTN4, CDC42, RAC1, WASL, ACTR2, ...",
1,astrocyte_identity,5,5,0,1.000,"GFAP, AQP4, ALDH1L1, SLC1A2, SLC1A3",
2,autophagy_stress,4,4,0,1.000,"FUNDC1, ATG4D, ATG9A, ATG7",
3,ev_markers,6,6,0,1.000,"CD9, CD37, CD63, CD81, CD82, FLOT1",
4,mdv_lysosomal,4,4,0,1.000,"LAMP1, LAMP2, RAB7, SNX9",
5,metabolic_coupling,4,4,0,1.000,"SLC16A1, SLC16A3, LDHA, LDHB",
6,mitochondrial_biogenesis,4,4,0,1.000,"PPARGC1A, TFAM, NRF1, NFE2L2",
7,mitochondrial_dynamics,5,5,0,1.000,"MFN1, MFN2, OPA1, DNM1L, FIS1",
8,mitochondrial_trafficking,6,6,0,1.000,"RHOT1, RHOT2, TRAK1, TRAK2, KIF5B, DYNC1H1",
9,mitophagy_stress,8,8,0,1.000,"PINK1, PRKN, BNIP3, BNIP3L, HIF1A, IL6, TNF, S...",


,gene,resolved,present
0,GJC2,Gjc2,True
1,GJB1,Gjb1,True
2,GJA1,Gja1,True


PosixPath('/Users/chrislangseth/work/karolinska_institutet/projects/cx47-oligo/results/04_transglial_mito_communication/lpc_dbit_multi/run_metadata.json')

## Helpers

In [ ]:
def score_all_panels(adata_input, panels):
    scored = {}
    for panel in panels:
        try:
            score, _ = score_gene_panel(adata_input, panel, normalize_counts=False)
            adata_input.obs[score.name] = score
            scored[panel] = score.name
        except ValueError:
            pass
    return scored


def spearman_by_day(frame, day_col, x_col, y_col, min_n=30):
    rows = []
    for day, sub in frame.groupby(day_col, observed=True):
        valid = sub[[x_col, y_col]].dropna()
        if len(valid) < min_n or valid[x_col].nunique() < 2 or valid[y_col].nunique() < 2:
            rows.append({"day": day, "n": len(valid), "rho": np.nan, "pvalue": np.nan})
            continue
        rho, pvalue = spearmanr(valid[x_col], valid[y_col])
        rows.append({"day": day, "n": len(valid), "rho": round(rho, 3), "pvalue": round(pvalue, 4)})
    return pd.DataFrame(rows)


def plot_heatmap(frame, title, cmap="mako"):
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(frame.columns) + 2), max(4, 0.5 * len(frame.index) + 2)))
    sns.heatmap(frame, cmap=cmap, linewidths=0.4, linecolor="white", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()


def plot_corr_trajectory(corr_rows, day_order, x_label, title):
    if not corr_rows:
        return
    df = pd.concat(corr_rows, ignore_index=True)
    df["day"] = pd.Categorical(df["day"], categories=day_order, ordered=True)
    fig, ax = plt.subplots(figsize=(max(7, 1.4 * len(day_order) + 3), 5))
    sns.lineplot(data=df, x="day", y="rho", hue="y_panel", marker="o", sort=False, ax=ax)
    ax.axhline(0, color="#666", linewidth=1, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Spearman rho")
    ax.legend(title=x_label, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.xticks(rotation=0)
    plt.tight_layout()

: 

## Score All Panels Across the Combined DBiT Dataset

In [ ]:
all_scored = score_all_panels(adata, ALL_PANELS)
print(f"Successfully scored {len(all_scored)}/{len(ALL_PANELS)} panels")

# Connexin expression
cx_expr = expression_frame(adata, TARGET_GENES, normalize_counts=False)
for gene in cx_expr.columns:
    adata.obs[f"{gene}_expr"] = cx_expr[gene].values

# Composite mito recovery score
mito_score_cols = [f"{p}_score" for p in MITO_PANELS if f"{p}_score" in adata.obs.columns]
if mito_score_cols:
    scaled = adata.obs[mito_score_cols].apply(
        lambda s: (s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) else 1.0)
    )
    adata.obs["composite_mito_score"] = scaled.mean(axis=1)

# Composite transfer score
transfer_score_cols = [f"{p}_score" for p in TRANSFER_PANELS if f"{p}_score" in adata.obs.columns]
if transfer_score_cols:
    scaled_t = adata.obs[transfer_score_cols].apply(
        lambda s: (s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) else 1.0)
    )
    adata.obs["composite_transfer_score"] = scaled_t.mean(axis=1)

print("Obs columns added:", [c for c in adata.obs.columns if "_score" in c or "_expr" in c])

## Identify Oligodendrocyte-Enriched Spots

Using the oligodendrocyte identity score relative to astrocyte and microglial scores, we tag spots as oligo-dominant, astro-dominant, or microglia-dominant.

In [ ]:
lineage_score_cols = [f"{p}_score" for p in LINEAGE_PANELS if f"{p}_score" in adata.obs.columns]

if lineage_score_cols:
    adata.obs["dominant_lineage"] = adata.obs[lineage_score_cols].idxmax(axis=1).str.replace("_score", "")
    lineage_counts = adata.obs.groupby(["timepoint_label", "dominant_lineage"], observed=True).size().unstack(fill_value=0)
    lineage_counts = lineage_counts.reindex(TIME_ORDER)
    display(lineage_counts)
    save_table(lineage_counts, RESULTS_DIR / "dominant_lineage_counts_by_day.csv")
    plot_heatmap(np.log1p(lineage_counts), "log1p spot counts by dominant lineage and day", cmap="Blues")
    #save_current_figure(FIGURES_DIR / "dominant_lineage_counts_heatmap.png")

    oligo_cutoff = adata.obs["oligodendrocyte_identity_score"].median() if "oligodendrocyte_identity_score" in adata.obs.columns else 0
    adata.obs["is_oligo_spot"] = (
        (adata.obs["dominant_lineage"] == "oligodendrocyte_identity")
        & (adata.obs.get("oligodendrocyte_identity_score", 0) >= oligo_cutoff)
    )
    adata.obs["is_astro_spot"] = (adata.obs["dominant_lineage"] == "astrocyte_identity")
    adata.obs["is_microglia_spot"] = (adata.obs["dominant_lineage"] == "microglial_activation")

    oligo_counts_by_day = adata.obs.groupby("timepoint_label", observed=True)["is_oligo_spot"].sum().reindex(TIME_ORDER)
    display(oligo_counts_by_day.rename("n_oligo_spots").to_frame())
else:
    print("Lineage panel scoring failed; cannot classify spots.")
    adata.obs["is_oligo_spot"] = True  # fallback: use all spots

## Cx47 vs Mitochondrial Programs in Oligo-Enriched Spots

Primary replication of the Q2 question at higher granularity: does GJC2 correlate with the mitochondrial program composite in spots classified as oligodendrocyte-dominant, and does this coupling change across days?

In [ ]:
adata_oligo_spots = adata[adata.obs["is_oligo_spot"]].copy()

q4_frame = adata_oligo_spots.obs[
    ["timepoint_label", "day_post_lpc", "GJC2_expr", "composite_mito_score"]
    + mito_score_cols
    + transfer_score_cols
].copy().dropna(subset=["GJC2_expr"])

# Day-level means
day_means = q4_frame.groupby("timepoint_label", observed=True).mean(numeric_only=True).reindex(TIME_ORDER)
display(day_means.round(3))
save_table(day_means, RESULTS_DIR / "oligo_spot_day_means.csv")

plot_heatmap(
    day_means[["GJC2_expr", "composite_mito_score"] + [c for c in mito_score_cols if c in day_means.columns]],
    "Oligo-spot mean scores across LPC days",
    cmap="crest",
)
#save_current_figure(FIGURES_DIR / "oligo_spot_day_means_heatmap.png")

# Spearman: GJC2 vs each mito panel, per day
corr_rows_mito = []
for panel_col in ["composite_mito_score"] + mito_score_cols:
    if panel_col not in q4_frame.columns:
        continue
    df_corr = spearman_by_day(q4_frame, "timepoint_label", "GJC2_expr", panel_col)
    df_corr["y_panel"] = panel_col.replace("_score", "")
    corr_rows_mito.append(df_corr)

if corr_rows_mito:
    mito_corr_table = pd.concat(corr_rows_mito, ignore_index=True)
    display(mito_corr_table.round(3))
    save_table(mito_corr_table, RESULTS_DIR / "oligo_spot_gjc2_mito_correlations_by_day.csv", index=False)
    plot_corr_trajectory(corr_rows_mito, TIME_ORDER, "Mito panel", "Spearman rho: GJC2 vs mitochondrial panels in oligo spots")
    #save_current_figure(FIGURES_DIR / "oligo_spot_gjc2_mito_corr_trajectory.png")

## Transglial Coupling: Oligodendrocyte Mito Score vs Neighbouring Glial Programs

In DBiT spatial data, if a spot has high oligo identity but also carries non-zero astrocyte or microglial program signal, this co-expression in the same spot is a spatial proxy for cellular proximity and potential metabolic coupling.

We ask: within oligo-dominant spots, does the mitochondrial composite score positively correlate with the astrocyte or microglial score? And does this relationship change across the LPC time axis?

In [ ]:
coupling_x_cols = ["GJC2_expr", "composite_mito_score"]
coupling_y_cols = [
    c for c in ["astrocyte_identity_score", "microglial_activation_score", "composite_transfer_score"]
    if c in q4_frame.columns
]

corr_rows_coupling = []
for x_col in coupling_x_cols:
    for y_col in coupling_y_cols:
        if x_col not in q4_frame.columns or y_col not in q4_frame.columns:
            continue
        df_corr = spearman_by_day(q4_frame, "timepoint_label", x_col, y_col)
        df_corr["x_panel"] = x_col.replace("_expr", "").replace("_score", "")
        df_corr["y_panel"] = y_col.replace("_score", "")
        corr_rows_coupling.append(df_corr)

if corr_rows_coupling:
    coupling_table = pd.concat(corr_rows_coupling, ignore_index=True)
    display(coupling_table.round(3))
    save_table(coupling_table, RESULTS_DIR / "transglial_coupling_correlations_by_day.csv", index=False)

    for x_col in coupling_x_cols:
        subset_rows = [r for r in corr_rows_coupling if r["x_panel"].iloc[0] == x_col.replace("_expr", "").replace("_score", "")]
        if subset_rows:
            plot_corr_trajectory(
                subset_rows,
                TIME_ORDER,
                "Glial panel",
                f"Spearman rho: {x_col.replace('_expr','').replace('_score','')} vs glial programs (oligo spots)",
            )
            #save_current_figure(FIGURES_DIR / f"coupling_corr_{x_col}.png")

## Vesicle Trafficking, EV Markers, and TNT-Related Programs

Potential mechanisms of transglial mitochondrial communication include:
- **Tunneling nanotubes (TNTs):** TNFAIP2, RALA, actin cytoskeleton
- **Extracellular vesicles (EVs):** CD9, CD63, CD81, FLOT1
- **Mitochondria-derived vesicles (MDVs):** LAMP1, RAB7 (lysosomal routing)
- **SNARE-mediated vesicle fusion:** VAMP2-8, STX4

We ask whether these programs co-vary with GJC2 and composite mito scores across spots and days.

In [ ]:
transfer_feature_cols = [
    c for c in transfer_score_cols + ["composite_transfer_score"]
    if c in adata.obs.columns
]

if transfer_feature_cols:
    # Day-level means across all spots
    transfer_day_means = (
        adata.obs.groupby("timepoint_label", observed=True)[transfer_feature_cols]
        .mean()
        .reindex(TIME_ORDER)
    )
    display(transfer_day_means.round(3))
    save_table(transfer_day_means, RESULTS_DIR / "transfer_scores_by_day.csv")

    transfer_z = transfer_day_means.apply(
        lambda s: (s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) else 1.0)
    )
    plot_heatmap(transfer_z, "Transfer mechanism program scores by LPC day (z-scored)", cmap="rocket")
    #save_current_figure(FIGURES_DIR / "transfer_scores_by_day_zscored.png")

    # Correlate transfer programs with GJC2 in oligo spots by day
    transfer_corr_rows = []
    for tc in transfer_feature_cols:
        if tc not in q4_frame.columns:
            continue
        df_corr = spearman_by_day(q4_frame, "timepoint_label", "GJC2_expr", tc)
        df_corr["y_panel"] = tc.replace("_score", "")
        transfer_corr_rows.append(df_corr)

    if transfer_corr_rows:
        transfer_corr_table = pd.concat(transfer_corr_rows, ignore_index=True)
        display(transfer_corr_table.round(3))
        save_table(transfer_corr_table, RESULTS_DIR / "transfer_gjc2_correlations_by_day.csv", index=False)
        plot_corr_trajectory(
            transfer_corr_rows, TIME_ORDER,
            "Transfer mechanism",
            "Spearman rho: GJC2 vs intercellular transfer programs (oligo spots by day)",
        )
        #save_current_figure(FIGURES_DIR / "transfer_gjc2_corr_trajectory.png")

## Summary: All Pairwise Correlations at Peak and Trough Days

Produce a global correlation matrix for all scored features within oligo-dominant spots, split by day, to identify which combinations are strongest and when.

In [ ]:
summary_cols = (
    ["GJC2_expr", "composite_mito_score", "composite_transfer_score"]
    + [c for c in mito_score_cols if c in q4_frame.columns]
    + [c for c in coupling_y_cols if c in q4_frame.columns]
)
summary_cols = [c for c in dict.fromkeys(summary_cols) if c in q4_frame.columns]

for day in TIME_ORDER:
    day_sub = q4_frame.loc[q4_frame["timepoint_label"] == day, summary_cols].dropna()
    if day_sub.shape[0] < 10:
        continue
    corr_mat = day_sub.corr(method="spearman")
    plot_heatmap(corr_mat, f"Spot-level Spearman correlations — oligo spots, {day}", cmap="RdBu_r")
    save_table(corr_mat, RESULTS_DIR / f"oligo_spot_corr_matrix_{day.replace(' ','')}.csv")
    #save_current_figure(FIGURES_DIR / f"oligo_spot_corr_matrix_{day.replace(' ','')}.png")